# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import torch
import tqdm

from experiments.models import (
    HyperTreeARForecast,
    HyperTreeNetARForecast,
)

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
    save_plot,
)

from plotnine import (
    ggplot, aes, geom_line, scale_color_manual, scale_size_manual,
    scale_linetype_manual,
    theme_bw, theme, element_line, element_blank, element_rect, labs,
    scale_x_continuous
)

repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / "plots"
result_path.mkdir(parents=True, exist_ok=True)

# Load Experiment Specifications

In [ ]:
# Load specifications
train_type = "global"
dataset = "rossmann"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]
params_lgb = config["lgb_params"]

htar_params = config["hyper_tree_params"]

htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# TS-Features

In [ ]:
# Extract time series features
ts_feats_df, ts_feats = create_tsfeatures(
    train=train_df,
    freq=freq
)

# Add features to train and test
train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
features = features + ts_feats

# Loop over Lags

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# Boosting Iterations (since we use relative runtimes per iteration, the absolute number is not important, but we want to ensure that it's sufficient to capture differences in runtime scaling across different lags)
num_boost_rounds = 30
htar_params["num_boost_round"] = num_boost_rounds
htnetar_params["num_boost_round"] = num_boost_rounds

# Specify Lags
lags = np.arange(10, 60, 10).reshape(-1,1)
lags = np.concatenate([np.array([np.arange(1, 6)]).T, lags], axis=0)

runtime_per_iter = []
for lag in tqdm.tqdm(lags, total=len(lags)):

    # Hyper-Tree-AR
    ht_ar_runtime = HyperTreeARForecast(
        htar_params,
        train_df,
        test_df.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        lag[0],
        loss_fn,
        seed
    )
    ht_ar_runtime = ht_ar_runtime.groupby("model").apply(lambda x: x["runtime"].unique()[0] / num_boost_rounds).rename("runtime_per_iter").reset_index()
    ht_ar_runtime["model"] = "Hyper-Tree-AR(p)"
    ht_ar_runtime["n_params"] = lag

    # Hyper-TreeNet-AR
    htnet_ar_runtime = HyperTreeNetARForecast(
        htnetar_params,
        htnetar_params_lgb,
        htnetar_network_params,
        train_df,
        test_df.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        lag[0],
        loss_fn,
        seed,
        device,
    )
    htnet_ar_runtime = htnet_ar_runtime.groupby("model").apply(lambda x: x["runtime"].unique()[0] / num_boost_rounds).rename("runtime_per_iter").reset_index()
    htnet_ar_runtime["model"] = "Hyper-TreeNet-AR(p)"
    htnet_ar_runtime["n_params"] = lag

    # Store results
    runtime_per_iter.append(
        pd.concat([ht_ar_runtime,htnet_ar_runtime], axis=0, ignore_index=True)
    )

# Runtimes

In [ ]:
time_per_iter_df = pd.concat(runtime_per_iter, axis=0).reset_index(drop=True)

# Plot

In [ ]:
# Prepare data for plotting
plot_df = (
    time_per_iter_df
    .groupby(["model", "n_params"])["runtime_per_iter"]
    .mean()
    .reset_index()
)

base_n = np.min(lags)
base_vals = (
    plot_df[plot_df["n_params"] == base_n]
    .copy()
    .drop(columns=["n_params"])
    .rename(columns={"runtime_per_iter": "base_time_per_iter"})
)

plot_df = plot_df.merge(base_vals, on="model", how="left")
plot_df["time_per_iter_rel"] = plot_df["runtime_per_iter"] / plot_df["base_time_per_iter"]

# Style mappings
color_map = {
    "Hyper-Tree-AR(p)": "#d62728",
    "Hyper-TreeNet-AR(p)": "#2ca02c",
}

size_map = {
    "Hyper-Tree-AR(p)": 2.0,
    "Hyper-TreeNet-AR(p)": 2.0,
}

linetype_map = {
    "Hyper-Tree-AR(p)": "solid",
    "Hyper-TreeNet-AR(p)": "solid",
}


xmax = int(plot_df["n_params"].max())
xticks = [1] + list(range(10, xmax + 1, 10))

# Create plot
scaling_plot = (
    ggplot(
        plot_df,
        aes(
            x="n_params",
            y="time_per_iter_rel",
            color="model",
            size="model",
            linetype="model"
        )
    )
    + geom_line()
    + scale_color_manual(values=color_map)
    + scale_size_manual(values=size_map)
    + scale_linetype_manual(values=linetype_map)
    + theme_bw(base_size=40)
    + theme(
        figure_size=(28, 15),
        legend_position="bottom",
        legend_title=element_blank(),
        legend_key=element_rect(fill="none", colour="none"),
        legend_margin=6,
        panel_grid_major_x=element_line(color="grey", alpha=0.05),
        panel_grid_minor_x=element_line(color="grey", alpha=0.05),
        panel_grid_major_y=element_line(color="grey", alpha=0.05),
        panel_grid_minor_y=element_line(color="grey", alpha=0.05),
    )
    + labs(
        title="",
        x="Number of Parameters (p)",
        y="Relative Runtime per Boosting Iteration",
    )
    + scale_x_continuous(breaks=xticks)
)

save_plot(scaling_plot, "runtime_scaling", result_path)
display(scaling_plot)